# Catan Game Logs Data Pipeline
This notebook reads raw game log HTML fragments and extracts structured data using Polars.

In [123]:
import polars as pl
from bs4 import BeautifulSoup
import re
import os
import glob
from tqdm import tqdm
from dataclasses import dataclass
import logging

logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='parser_logs.txt',
    filemode='w'
)


## Parser Functions

In [124]:
@dataclass(frozen=True)
class ActionType:
    END_OF_TURN = 'end_of_turn'
    PLACED_SETTLEMENT = 'placed_settlement'
    PLACED_ROAD = 'placed_road'
    BUILT_SETTLEMENT = 'built_settlement'
    BUILT_CITY = 'built_city'
    BUILT_ROAD = 'built_road'
    STARTING_RESOURCES = 'starting_resources'
    ROLLED_DICE = 'rolled_dice'
    MOVED_ROBBER = 'moved_robber'
    STOLE = 'stole'
    MONOPOLY_STEAL = 'monopoly_steal'
    PROPOSED_TRADE = 'proposed_trade'
    FRIENDLY_ROBBER = 'friendly_robber'
    ROBBER_BLOCKED = 'robber_blocked'
    GOT_RESOURCE = 'got_resource'
    COUNTER_OFFER = 'counter_offer'
    COMPLETED_TRADE = 'completed_trade'
    BANK_TRADE = 'bank_trade'
    BOUGHT_DEV_CARD = 'bought_dev_card'
    DISCARDED = 'discarded'
    PLAYED_KNIGHT = 'played_knight'
    PLAYED_MONOPOLY = 'played_monopoly'
    LONGEST_ROAD = 'longest_road'
    LARGEST_ARMY = 'largest_army'
    LONGEST_ROAD_TRANSFERRED = 'longest_road_transferred'
    BOT_DISCARDING = 'bot_discarding'
    NO_STEAL_TARGET = 'no_steal_target'
    WIN = 'win'
    BLOCKED_TRADING = 'blocked_trading'
    UNKNOWN = 'unknown'


In [125]:

def parse_log_line(html_string):
    soup = BeautifulSoup(html_string, 'html.parser')
    feed_msg = soup.find('div', class_='feedMessage-O8TLknGe')
    if not feed_msg:
        return None
    
    msg_part = feed_msg.find('span', class_='messagePart-XeUsOgLX')
    if not msg_part:
        return None
        
    text = msg_part.get_text()
    
    # Check if HR
    if msg_part.find('hr'):
        return {'action_type': ActionType.END_OF_TURN}
        
    action_data = {'raw_text': text}
    
    valid_resources = ["Wool", "Brick", "Lumber", "Ore", "Grain"]
    
    def extract_resources(container):
        return [img.get('alt') for img in container.find_all('img') if img.get('alt') in valid_resources or img.get('alt') in [r.lower() for r in valid_resources]]
    
    # Simple regex matches
    if 'placed a Settlement' in text:
        action_data['action_type'] = ActionType.PLACED_SETTLEMENT
        action_data['player'] = msg_part.find('span').text

    elif 'placed a Road' in text:
        action_data['action_type'] = ActionType.PLACED_ROAD
        action_data['player'] = msg_part.find('span').text

    elif 'built a Settlement' in text:
        action_data['action_type'] = ActionType.BUILT_SETTLEMENT
        action_data['player'] = msg_part.find('span').text

    elif 'built a City' in text:
        action_data['action_type'] = ActionType.BUILT_CITY
        action_data['player'] = msg_part.find('span').text

    elif 'built a Road' in text:
        action_data['action_type'] = ActionType.BUILT_ROAD
        action_data['player'] = msg_part.find('span').text

    elif 'received starting resources' in text:
        action_data['action_type'] = ActionType.STARTING_RESOURCES
        action_data['player'] = msg_part.find('span').text
        action_data['resources'] = extract_resources(msg_part)

    elif 'rolled' in text:
        action_data['action_type'] = ActionType.ROLLED_DICE
        action_data['player'] = msg_part.find('span').text
        dice = [int(img.get('alt').split('_')[1]) for img in msg_part.find_all('img') if 'dice_' in img.get('alt', '')]
        action_data['roll_total'] = sum(dice) if dice else None

    elif 'moved Robber' in text:
         action_data['action_type'] = ActionType.MOVED_ROBBER
         action_data['player'] = msg_part.find('span').text

    elif 'stole' in text and 'from' in text:
        if text.startswith('You stole'):
            action_data['action_type'] = ActionType.STOLE
            action_data['player'] = 'You'
            action_data['target'] = msg_part.find_all('span')[-1].text

        elif 'stole 2' in text or 'stole 1' in text or 'stole' in text:
            # Need to distinguish between normal steal and monopoly steal
            if 'stole' in text and len(msg_part.find_all('span')) >= 2:
                 action_data['action_type'] = ActionType.STOLE
                 action_data['player'] = msg_part.find_all('span')[0].text
                 action_data['target'] = msg_part.find_all('span')[-1].text
            else:
                 action_data['action_type'] = ActionType.MONOPOLY_STEAL
                 action_data['player'] = msg_part.find_all('span')[0].text
                 m = re.search(r'stole (\d+)', text)
                 action_data['amount'] = int(m.group(1)) if m else 1
                 # Fallback alt checks
                 action_data['stolen_resource'] = extract_resources(msg_part)[0] if extract_resources(msg_part) else None

    elif 'wants to give' in text and 'for' in text:
        action_data['action_type'] = ActionType.PROPOSED_TRADE
        action_data['player'] = msg_part.find('span').text

    elif 'Friendly Robber is active' in text:
        action_data['action_type'] = ActionType.FRIENDLY_ROBBER

    elif 'blocked by the Robber' in text:
        action_data['action_type'] = ActionType.ROBBER_BLOCKED

    elif 'got' in text and not 'gave' in text and not 'discarded' in text:
        # Avoid matching 'gave X and got Y'
        action_data['action_type'] = ActionType.GOT_RESOURCE
        if msg_part.find('span'):
            action_data['player'] = msg_part.find('span').text
            action_data['resources'] = extract_resources(msg_part)

    elif 'proposed counter offer' in text:
        action_data['action_type'] = ActionType.COUNTER_OFFER
        spans = msg_part.find_all('span')
        if len(spans) >= 2:
            action_data['player'] = spans[0].text
            action_data['target'] = spans[1].text

    elif 'gave' in text and 'and got' in text and 'from' in text:
         action_data['action_type'] = ActionType.COMPLETED_TRADE
         spans = msg_part.find_all('span')
         if len(spans) >= 2:
            action_data['player'] = spans[0].text
            action_data['target'] = spans[-1].text
            
            # The text layout is usually: `Player` gave `<imgs>` and got `<imgs>` from `Target`
            # Let's cleanly split using beautiful soup nodes to know which is which. 
            parts = list(msg_part.children)
            gave_imgs = []
            got_imgs = []
            current_list = gave_imgs
            
            for child in parts:
                if isinstance(child, str):
                    if 'got' in child:
                        current_list = got_imgs
                elif child.name == 'img':
                    alt = child.get('alt')
                    if alt in valid_resources or alt in [r.lower() for r in valid_resources]:
                        current_list.append(alt)
                        
            action_data['gave_resources'] = gave_imgs
            action_data['got_resources'] = got_imgs
            action_data['cards_lost'] = len(gave_imgs)
            action_data['cards_gained'] = len(got_imgs)

    elif 'gave bank' in text and 'and took' in text:
         action_data['action_type'] = ActionType.BANK_TRADE
         action_data['player'] = msg_part.find('span').text
         
         parts = list(msg_part.children)
         gave_imgs = []
         got_imgs = []
         current_list = gave_imgs
         
         for child in parts:
             if isinstance(child, str):
                 if 'took' in child:
                     current_list = got_imgs
             elif child.name == 'img':
                 alt = child.get('alt')
                 if alt in valid_resources or alt in [r.lower() for r in valid_resources]:
                     current_list.append(alt)
                     
         action_data['gave_resources'] = gave_imgs
         action_data['got_resources'] = got_imgs

    elif 'bought' in text and 'Development Card' in str(msg_part):
         action_data['action_type'] = ActionType.BOUGHT_DEV_CARD
         action_data['player'] = msg_part.find('span').text

    elif 'discarded' in text:
         action_data['action_type'] = ActionType.DISCARDED
         action_data['player'] = msg_part.find('span').text
         resources = extract_resources(msg_part)
         action_data['discard_count'] = len(resources)
         action_data['discarded_resources'] = resources

    elif 'used' in text and 'Knight' in str(feed_msg):
        action_data['action_type'] = ActionType.PLAYED_KNIGHT
        action_data['player'] = msg_part.find('span').text

    elif 'used' in text and 'Monopoly' in str(feed_msg):
        action_data['action_type'] = ActionType.PLAYED_MONOPOLY
        action_data['player'] = msg_part.find('span').text

    elif 'received Longest Road' in text:
        action_data['action_type'] = ActionType.LONGEST_ROAD
        action_data['player'] = msg_part.find('span').text

    elif 'received Largest Army' in text:
        action_data['action_type'] = ActionType.LARGEST_ARMY
        action_data['player'] = msg_part.find('span').text

    elif 'Longest Road' in text and 'passed from' in text:
        action_data['action_type'] = ActionType.LONGEST_ROAD_TRANSFERRED

    elif 'selecting cards to discard' in text:
        action_data['action_type'] = ActionType.BOT_DISCARDING

    elif 'No player to steal from' in text:
        action_data['action_type'] = ActionType.NO_STEAL_TARGET

    elif 'won the game' in text:
        action_data['action_type'] = ActionType.WIN
        action_data['player'] = msg_part.find('span').text

    elif 'blocked trading with' in text:
        action_data['action_type'] = ActionType.BLOCKED_TRADING
        
    else:
        action_data['action_type'] = ActionType.UNKNOWN
        
    logging.info(f"Parsed action: {action_data}")
    return action_data


## Load and Process Data

In [126]:
def process_game_file(filepath):
    game_id = os.path.basename(filepath).split('_')[2].split('.')[0]
    
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split the content into divs roughly (each div is separated by double newlines usually)
    # Using BeautifulSoup directly on the whole file or splitting by <div data-index
    divs = re.split(r'\n(?=<div data-index)', content)
    
    actions = []
    turn_counter = 0
    
    for div_html in divs:
        if not div_html.strip():
            continue
            
        parsed = parse_log_line(div_html)
        if parsed:
            parsed['game_id'] = game_id
            parsed['turn_number'] = turn_counter
            
            if parsed['action_type'] == ActionType.END_OF_TURN:
                turn_counter += 1
                
            actions.append(parsed)
            
    # Resolve "You" to the correct player name
    players_in_game = set()
    for a in actions:
        if a.get('player') and a['player'] != 'You':
            players_in_game.add(a['player'])
        if a.get('target') and a['target'] != 'You':
            players_in_game.add(a['target'])
            
    identity_map = ["Romeoore", "UniQueLagacy", "HomeofAD3005", "AatNeverLose"]
    you_player = next((p for p in identity_map if p in players_in_game), None)
    
    if you_player:
        for a in actions:
            if a.get('player') == 'You':
                a['player'] = you_player
            if a.get('target') == 'You':
                a['target'] = you_player
                
    return actions

all_actions = []
data_dir = '../../data/raw_divs'
file_paths = glob.glob(os.path.join(data_dir, '*.txt'))

for fp in tqdm(file_paths):
    all_actions.extend(process_game_file(fp))
    
# Convert to Polars DataFrame
df = pl.DataFrame(all_actions)


100%|██████████| 28/28 [00:02<00:00, 12.55it/s]


## Calculate Stats per Game

In [127]:
# View some of the parsed data
print(df.head())
print("Unique action types extracted:", df['action_type'].unique().to_list())


shape: (5, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ raw_text  ┆ action_ty ┆ game_id   ┆ turn_numb ┆ … ┆ gave_reso ┆ got_resou ┆ cards_los ┆ cards_ga │
│ ---       ┆ pe        ┆ ---       ┆ er        ┆   ┆ urces     ┆ rces      ┆ t         ┆ ined     │
│ str       ┆ ---       ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ str       ┆           ┆ i64       ┆   ┆ list[str] ┆ list[str] ┆ i64       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Happy     ┆ unknown   ┆ 192053656 ┆ 0         ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│ settling! ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ Learn how ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ to p…     ┆           ┆           ┆           ┆   ┆           ┆           

In [128]:
def calculate_game_stats(df_game):
    # Get unique players
    players = df_game.filter(pl.col('player').is_not_null())['player'].unique().to_list()
    players = [p for p in players if p != 'You']
    
    # -------------------------------------------------------------
    # 1. Track Hand Sizes chronologically to get "Average Cards Held"
    # -------------------------------------------------------------
    hand_sizes = {p: 0 for p in players}
    turn_hand_sizes = {p: [] for p in players}
    
    for row in df_game.iter_rows(named=True):
        act = row.get('action_type')
        p = row.get('player')
        t = row.get('target')

        deltas = {p_name: 0 for p_name in players}

        if act == ActionType.STARTING_RESOURCES and p in players:
            res = row.get('resources')
            deltas[p] += len(res) if res else 0
        elif act == ActionType.BUILT_SETTLEMENT and p in players:
            deltas[p] -= 4
        elif act == ActionType.BUILT_CITY and p in players:
            deltas[p] -= 5
        elif act == ActionType.BUILT_ROAD and p in players:
            deltas[p] -= 2
        elif act == ActionType.BOUGHT_DEV_CARD and p in players:
            deltas[p] -= 3
        elif act == ActionType.GOT_RESOURCE and p in players:
            res = row.get('resources')
            deltas[p] += len(res) if res else 0
        elif act == ActionType.COMPLETED_TRADE:
            cg = row.get('cards_gained') or 0
            cl = row.get('cards_lost') or 0
            if p in players:
                deltas[p] += cg - cl
            if t in players: 
                deltas[t] += cl - cg
        elif act == ActionType.BANK_TRADE and p in players:
            gives = row.get('gave_resources')
            gots = row.get('got_resources')
            deltas[p] += (len(gots) if gots else 0) - (len(gives) if gives else 0)
        elif act == ActionType.STOLE:
            if p in players: deltas[p] += 1
            if t in players: deltas[t] -= 1
        elif act == ActionType.MONOPOLY_STEAL and p in players:
            deltas[p] += row.get('amount') or 0
        elif act == ActionType.DISCARDED and p in players:
            deltas[p] -= row.get('discard_count') or 0

        # Apply deltas
        for p_name, d in deltas.items():
            hand_sizes[p_name] = max(0, hand_sizes[p_name] + d) # Prevent negative hands from missed parses

        # Record hand size at end of each turn
        if act == ActionType.END_OF_TURN:
            for p_name in players:
                turn_hand_sizes[p_name].append(hand_sizes[p_name])
                
    # -------------------------------------------------------------
    # 2. Extract Individual Player Stats
    # -------------------------------------------------------------
    stats = []
    for player in players:
        p_df = df_game.filter(pl.col('player') == player)
        target_df = df_game.filter(pl.col('target') == player)
        
        # Calculate cards lost / gained in trades
        completed_trades = p_df.filter(pl.col('action_type') == ActionType.COMPLETED_TRADE)
        cards_lost_in_trades = completed_trades['cards_lost'].sum() if 'cards_lost' in completed_trades.columns else 0
        cards_gained_in_trades = completed_trades['cards_gained'].sum() if 'cards_gained' in completed_trades.columns else 0
        
        completed_trades_as_target = target_df.filter(pl.col('action_type') == ActionType.COMPLETED_TRADE)
        if len(completed_trades_as_target) > 0 and 'cards_gained' in completed_trades_as_target.columns:
            cards_lost_in_trades += completed_trades_as_target['cards_gained'].sum() 
            cards_gained_in_trades += completed_trades_as_target['cards_lost'].sum() 
            
        # Count Resources discarded total
        discarded_actions = p_df.filter(pl.col('action_type') == ActionType.DISCARDED)
        resources_lost_to_7_roll = discarded_actions['discard_count'].sum() if 'discard_count' in discarded_actions.columns else 0
        
        # Calculate turns before FIRST trade proposed by THIS player
        player_proposed_trades = p_df.filter(pl.col('action_type') == ActionType.PROPOSED_TRADE)
        turns_before_first_trade = player_proposed_trades['turn_number'][0] if len(player_proposed_trades) > 0 else -1

        # Count starting resources
        start_res = {'ore': 0, 'wool': 0, 'lumber': 0, 'grain': 0, 'brick': 0}
        if 'resources' in p_df.columns:
            start_actions = p_df.filter(pl.col('action_type') == ActionType.STARTING_RESOURCES)
            for r_list in start_actions['resources'].to_list():
                if r_list:
                    for r in r_list:
                        if r and r.lower() in start_res:
                            start_res[r.lower()] += 1

        # Generic Resource Counter
        res_gained = {'ore': 0, 'wool': 0, 'lumber': 0, 'grain': 0, 'brick': 0}
        def count_resources(res_list, track_dict=res_gained):
            if res_list is None: return
            for r in res_list:
                if not r: continue
                r_lower = r.lower()
                if r_lower in track_dict:
                    track_dict[r_lower] += 1
                
        # Scarcity Patterns: Resources explicitly traded away and received
        traded_away = {'ore': 0, 'wool': 0, 'lumber': 0, 'grain': 0, 'brick': 0}
        received_in_trade = {'ore': 0, 'wool': 0, 'lumber': 0, 'grain': 0, 'brick': 0}

        # Completed trades (as initiator)
        if 'gave_resources' in p_df.columns and 'got_resources' in p_df.columns:
            for row in p_df.filter(pl.col('action_type') == ActionType.COMPLETED_TRADE).iter_rows(named=True):
                count_resources(row.get('gave_resources'), traded_away)
                count_resources(row.get('got_resources'), received_in_trade)
                
        # Completed trades (as target)
        if 'gave_resources' in target_df.columns and 'got_resources' in target_df.columns:
            for row in target_df.filter(pl.col('action_type') == ActionType.COMPLETED_TRADE).iter_rows(named=True):
                count_resources(row.get('got_resources'), traded_away) 
                count_resources(row.get('gave_resources'), received_in_trade)
                
        # Bank trades
        if 'gave_resources' in p_df.columns and 'got_resources' in p_df.columns:
            for row in p_df.filter(pl.col('action_type') == ActionType.BANK_TRADE).iter_rows(named=True):
                count_resources(row.get('gave_resources'), traded_away)
                count_resources(row.get('got_resources'), received_in_trade)
        
        # Starting and general got resources
        for col_name in ['resources']:
            if col_name in p_df.columns:
                for r_list in p_df.filter(pl.col('action_type').is_in([ActionType.STARTING_RESOURCES, ActionType.GOT_RESOURCE]))[col_name].to_list():
                    count_resources(r_list)
                    
        # Monopoly steals gained
        if 'stolen_resource' in p_df.columns and 'amount' in p_df.columns:
            mono_steals = p_df.filter(pl.col('action_type') == ActionType.MONOPOLY_STEAL)
            for row in mono_steals.iter_rows(named=True):
                stolen = row.get('stolen_resource')
                if stolen:
                    amt = row.get('amount') or 1
                    count_resources([stolen] * amt)
                    
        # Calculate Average Hand Size limit
        hand_history = turn_hand_sizes[player]
        avg_hand_size = sum(hand_history) / len(hand_history) if len(hand_history) > 0 else 0

        stat = {
            'player': player,
            'settlements_built': p_df.filter(pl.col('action_type').is_in([ActionType.PLACED_SETTLEMENT, ActionType.BUILT_SETTLEMENT])).height,
            'cities_built': p_df.filter(pl.col('action_type') == ActionType.BUILT_CITY).height,
            'roads_built': p_df.filter(pl.col('action_type').is_in([ActionType.PLACED_ROAD, ActionType.BUILT_ROAD])).height,
            'bank_trades': p_df.filter(pl.col('action_type') == ActionType.BANK_TRADE).height,
            'trades_proposed': p_df.filter(pl.col('action_type') == ActionType.PROPOSED_TRADE).height,
            'trades_completed': p_df.filter(pl.col('action_type') == ActionType.COMPLETED_TRADE).height,
            'counter_offers': p_df.filter(pl.col('action_type') == ActionType.COUNTER_OFFER).height,
            'dev_cards_bought': p_df.filter(pl.col('action_type') == ActionType.BOUGHT_DEV_CARD).height,
            'times_discarded': p_df.filter(pl.col('action_type') == ActionType.DISCARDED).height,
            
            'cards_lost_in_trades': cards_lost_in_trades,
            'cards_gained_in_trades': cards_gained_in_trades,
            'resources_lost_to_7_roll': resources_lost_to_7_roll,
            
            'avg_hand_size': avg_hand_size,
            
            # Scarcity / Trading Metrics
            'traded_away_ore': traded_away['ore'],
            'traded_away_wool': traded_away['wool'],
            'traded_away_lumber': traded_away['lumber'],
            'traded_away_grain': traded_away['grain'],
            'traded_away_brick': traded_away['brick'],

            'received_trade_ore': received_in_trade['ore'],
            'received_trade_wool': received_in_trade['wool'],
            'received_trade_lumber': received_in_trade['lumber'],
            'received_trade_grain': received_in_trade['grain'],
            'received_trade_brick': received_in_trade['brick'],
            
            # General Accumulation
            'ore_gained_total': res_gained['ore'],
            'wool_gained_total': res_gained['wool'],
            'lumber_gained_total': res_gained['lumber'],
            'grain_gained_total': res_gained['grain'],
            'brick_gained_total': res_gained['brick'],
            
            'starting_ore': start_res['ore'],
            'starting_wool': start_res['wool'],
            'starting_lumber': start_res['lumber'],
            'starting_grain': start_res['grain'],
            'starting_brick': start_res['brick'],
            
            'longest_road_received': p_df.filter(pl.col('action_type') == ActionType.LONGEST_ROAD).height,
            'largest_army_received': p_df.filter(pl.col('action_type') == ActionType.LARGEST_ARMY).height,
            'unique_players_stolen_from': p_df.filter(pl.col('action_type') == ActionType.STOLE)['target'].n_unique() if 'target' in p_df.columns else 0,
            'times_targeted_by_others': target_df.filter(pl.col('action_type') == ActionType.STOLE).height,
            'turns_before_first_trade': turns_before_first_trade
        }
        stats.append(stat)
        
    return pl.DataFrame(stats)


In [129]:

# Iterate through each unique game_id, calculate stats, and export to Parquet
out_dir = '../../data/game_log_dfs'
os.makedirs(out_dir, exist_ok=True)

unique_games = df['game_id'].unique().to_list()
for game_id in tqdm(unique_games, desc="Exporting Game Stats"):
    df_game = df.filter(pl.col('game_id') == game_id)
    game_stats = calculate_game_stats(df_game)
    
    out_path = os.path.join(out_dir, f"game_{game_id}_stats.parquet")
    game_stats.write_parquet(out_path)

print(f"Exported stats for {len(unique_games)} games to {out_dir}")


Exporting Game Stats: 100%|██████████| 28/28 [00:00<00:00, 74.89it/s]

Exported stats for 28 games to ../../data/game_log_dfs
